In [1]:
# !pip install fusion_solar_py -q

In [2]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")
# LAT = float(user_secrets.get_secret("LAT"))
# LON = float(user_secrets.get_secret("LON"))

In [4]:
import os

from dotenv import load_dotenv

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")
LAT = float(os.environ.get("LAT"))
LON = float(os.environ.get("LON"))

In [5]:
import logging
from time import sleep
from datetime import timedelta, datetime

import numpy as np
import pandas as pd

from energymanagementrl.fusion_solar_connector import *
from energymanagementrl.production_forecast import *
from energymanagementrl.simulation import sparse_matrix
from energymanagementrl.rl import extract_values_gen
from stable_baselines3 import DQN

In [23]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Create a file handler for logging to a file
file_handler = logging.FileHandler('.log')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Create a console handler for logging to the screen
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Add both handlers to the logger

for handler in logger.handlers[:]:
    logger.removeHandler(handler)
logger.addHandler(file_handler)
logger.addHandler(console_handler)



In [3]:
_model = DQN.load(
    # '../logs/best_model/best_model.zip',
    '../logs/ppo_inverter.0.1.b',
    # '../logs/ppo_inverter.b.9.0',
)

In [7]:
panel_model = PanelModel(pdc0=0.42, temp_model_a=-3.56, temp_model_b=-0.075, delta_t=3, gamma_pdc=-0.004)
num_panels = 14
arrays = [ArrayConfig(name='sud_east', panel_model=panel_model, num_panels=num_panels, tilt_angle=25, azimuth=110),
          ArrayConfig(name='nord_west', panel_model=panel_model, num_panels=num_panels, tilt_angle=18, azimuth=290)]

_plant_config = PlantConfig(
    latitude=LAT, longitude=LON, timezone='Europe/Rome', inverter_pdc0=6, arrays=arrays
)
_production_forecaster = EnergyPredictionSystem(plant_config=_plant_config, open_meteo_client=OpenMeteoClient())

In [8]:
_client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                  huawei_subdomain="uni004eu5")
periodic_task = PeriodicTask(_client.keep_alive)
periodic_task.start()
_plant_id = _client.get_plant_ids()[0]
battery_id = _client.get_battery_ids(_plant_id)[0]

In [16]:
def get_flow_and_energy(
        client: FusionSolarClientParsed,
        plant_id: str,
        battery_capacity_kw: int,
        battery_min_percentage: int = 0
):
    prod_kw, load_kw, charge_kw, grid_kw, soc = client.get_plant_flow_parsed(plant_id)
    prod_kwh, load_kwh, charge_kwh, grid_kwh = [i / 12 for i in (prod_kw, -load_kw, charge_kw, grid_kw)]
    stored_kwh = (soc - battery_min_percentage) / 100 * battery_capacity_kw
    return prod_kwh, load_kwh, charge_kwh, grid_kwh, stored_kwh


def get_history(
        client: FusionSolarClientParsed,
        plant_id: str
):
    now = datetime.now()
    history = pd.concat([
        client.get_plant_stats_parsed(
            plant_id,
            query_time=client._get_day_start_sec() + i * client.MILLISECONDS_IN_A_DAY,
            time_zone=2,
            time_zone_str='Europe/Rome'
        )
        for i in [-2, -1, 0]
    ]).usePower
    history = history[history.index <= now][-288 * 2:].replace('--', np.nan)
    history = history.astype('float')
    if history.isnull().values.any():
        history.fillna(history.mean(), inplace=True)
    load_kwh_last_2day = history.resample('4h').mean()[1:] / 12
    return load_kwh_last_2day


def compute_production_residual(production_forecaster):
    now_gmt = pd.Timestamp.now(tz='UTC').to_pydatetime()
    now_gmt_5m = now_gmt.replace(minute=now_gmt.minute // 5 * 5)
    start = now_gmt_5m.strftime('%Y-%m-%d %H:%M')
    end = (now_gmt_5m + timedelta(days=2)).strftime('%Y-%m-%d %H:%M')
    clear_sky_df = production_forecaster.run_energy_production_prediction(start, end, WeatherType.clear_sky) / 12
    open_meteo_df = production_forecaster.run_energy_production_prediction(start, end,
                                                                           WeatherType.open_meteo_forecast) / 12
    forecast_range = [i * 12 for i in range(48)]
    residual = abs(clear_sky_df.inverter_ac - open_meteo_df.inverter_ac)
    prod_kwh_next_2day, residual_kwh_next_2day = (
        np.array(column.iloc[forecast_range]) @ sparse_matrix
        for column in (open_meteo_df.inverter_ac, residual)
    )
    return prod_kwh_next_2day, residual_kwh_next_2day


def build_system_state(prod_kwh, load_kwh, charge_kwh, grid_kwh, stored_kwh, load_kwh_last_2day,
                       prod_kwh_next_2day, residual_kwh_next_2day):
    state = {
        'prod_sim': {},
        'cons_sim': {},
        'batt_sim': {},
        'grid_sim': {},
    }

    for i, value in enumerate(prod_kwh_next_2day):
        state['prod_sim'][f"energy_sample_{i}"] = round(float(value), 3)
    state["prod_sim"]["energy"] = prod_kwh
    for i, value in enumerate(residual_kwh_next_2day):
        state['prod_sim'][f"residual_sample_{i}"] = round(float(value), 3)

    for i, value in enumerate(load_kwh_last_2day):
        state['cons_sim'][f"energy_sample_{i}"] = round(float(value), 3)
    state["cons_sim"]["energy"] = round(load_kwh, 3)

    state["batt_sim"]["stored"] = round(stored_kwh, 3)
    state["batt_sim"]["charge_rate"] = -round(charge_kwh, 3) if charge_kwh < 0 else 0
    state["batt_sim"]["discharge_rate"] = round(charge_kwh, 3) if charge_kwh > 0 else 0

    state["grid_sim"]["feed_to_grid"] = -round(grid_kwh, 3) if grid_kwh < 0 else 0
    state["grid_sim"]["taken_from_grid"] = round(grid_kwh, 3) if grid_kwh > 0 else 0

    return state


def get_system_state(client, plant_id, production_forecaster):
    prod_kwh, load_kwh, charge_kwh, grid_kwh, stored_kwh = get_flow_and_energy(client, plant_id, 10, 10)
    load_kwh_last_2day = get_history(client, plant_id)
    prod_kwh_next_2day, residual_kwh_next_2day = compute_production_residual(production_forecaster)

    return build_system_state(
        prod_kwh, load_kwh, charge_kwh, grid_kwh, stored_kwh, load_kwh_last_2day,
        prod_kwh_next_2day, residual_kwh_next_2day
    )

In [ ]:
def execute_control(client, plant_id, production_forecaster, model, active=False):
    """Executes control logic to adjust battery working mode based on model predictions."""

    # Retrieve system state
    state = get_system_state(client, plant_id, production_forecaster)
    obs = np.array(list(extract_values_gen(state)))

    # Validate observation length
    if len(obs) != 55:
        raise FusionSolarExceptionExtended(
            "Invalid observation length",
            FusionSolarExceptionExtended.ErrorCode.PARSING
        )

    # Predict action using the model
    action, _ = model.predict(obs)

    # Determine battery mode based on action
    battery_mode = (
        FusionSolarClientParsed.BatteryWorkingMode.MAXIMUM_SELF_CONSUMPTION
        if action == 1 else
        FusionSolarClientParsed.BatteryWorkingMode.FULLY_FEED_TO_GRID
    )

    # Set battery mode if active
    if active:
        client.set_battery_working_mode(battery_id, battery_mode)
    logging.warning(f"Battery Mode: {battery_mode.name}")

    return state, action


def main_loop():
    """Main loop for executing control at regular intervals."""
    try:
        while True:
            # Execute control logic
            try:
                state_, action_ = execute_control(
                    client=_client,
                    plant_id=_plant_id,
                    production_forecaster=_production_forecaster,
                    model=_model,
                    active=True
                )
            except FusionSolarExceptionExtended as e:
                logging.error(f"Error: {e.code}")
                sleep(10)
                continue
            # Log the state and action with a timestamp
            now = datetime.now()
            state_.update({
                'timestamp': now.timestamp(),
                'action': int(action_)
            })
            logging.info(f"State :{state_}")

            # Calculate the next execution time (aligned to the next 5-minute interval)
            next_time = (now + timedelta(minutes=5 - now.minute % 5)).replace(second=0, microsecond=0)
            sleep_duration = (next_time - now).total_seconds()

            # Sleep until the next interval
            sleep(sleep_duration)

    finally:
        # Reset battery working mode on exit
        _client.set_battery_working_mode(
            battery_id,
            FusionSolarClientParsed.BatteryWorkingMode.MAXIMUM_SELF_CONSUMPTION
        )
        logging.warning("Control loop terminated, battery mode reset.")


main_loop()

2024-12-19 08:00:18,618 - WARNING - Battery Mode: FULLY_FEED_TO_GRID
2024-12-19 08:05:02,716 - WARNING - Battery Mode: FULLY_FEED_TO_GRID
2024-12-19 08:10:03,006 - WARNING - Battery Mode: FULLY_FEED_TO_GRID
2024-12-19 08:15:03,310 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:20:02,564 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:25:02,911 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:30:03,047 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:35:02,774 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:40:02,906 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:45:03,145 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:50:02,971 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 08:55:02,495 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 09:00:03,139 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION
2024-12-19 09:05:03,073 - WARNING - Battery

In [ ]:
state = get_system_state(_client, _plant_id, _plant_config)
obs = np.array(list(extract_values_gen(state)))
# obs[-3]=200
# obs[-4]=0
# obs[-2]=200
# obs[-5] = 8800

if len(obs) == 55:
    action, _ = _model.predict(obs / 1000)
    print(int(action))
state